# Employee Attrition Prediction

## 📊 Business Context
Predict turnover.

**Analytical Approach:** Classification
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

In [ ]:
# Data Generation
def generate_data(n=1000):
    np.random.seed(42)
    # Generate synthetic features
    data = pd.DataFrame({
        'Satisfaction': np.random.normal(50, 15, n),
        'Tenure': np.random.exponential(10, n),
        'Salary': np.random.randint(0, 100, n),
        'Dept': np.random.choice(['A', 'B', 'C'], n, p=[0.5, 0.3, 0.2]),
        'Tenure': np.random.randint(1, 60, n),
        'Age': np.random.normal(35, 10, n)
    })
    
    # Introduce some missing values
    data.loc[np.random.choice(data.index, size=int(n*0.05)), 'Satisfaction'] = np.nan
    
    # Generate target with complex logic
    prob = (data['Satisfaction'].fillna(50)/100 + data['Salary']/200) / 2
    prob += np.where(data['Dept'] == 'C', 0.2, 0)
    prob -= data['Tenure'] / 100
    
    data['Left'] = (prob + np.random.normal(0, 0.1, n) > 0.55).astype(int)
    return data

df = generate_data(1500)
print(f'Dataset Shape: {df.shape}')
display(df.head())
display(df.describe())

In [ ]:
# Exploratory Data Analysis (EDA)
def perform_eda(df, target):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Target Distribution
    sns.countplot(x=target, data=df, ax=axes[0,0])
    axes[0,0].set_title('Target Distribution')
    
    # Numerical Feature Distribution
    sns.histplot(data=df, x='Satisfaction', hue=target, kde=True, ax=axes[0,1])
    axes[0,1].set_title('Feature 1 Distribution by Target')
    
    # Categorical Feature
    sns.countplot(x='Dept', hue=target, data=df, ax=axes[1,0])
    axes[1,0].set_title('Feature 4 vs Target')
    
    # Correlation Heatmap
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', ax=axes[1,1])
    axes[1,1].set_title('Correlation Matrix')
    
    plt.tight_layout()
    plt.show()

perform_eda(df, 'Left')

In [ ]:
# Preprocessing & Feature Engineering
class DataPreprocessor:
    def __init__(self, df, target):
        self.df = df.copy()
        self.target = target
        self.scaler = StandardScaler()
        self.imputer = SimpleImputer(strategy='median')
        
    def process(self):
        # Handle Missing Values
        num_cols = self.df.select_dtypes(include=[np.number]).columns.drop(self.target)
        self.df[num_cols] = self.imputer.fit_transform(self.df[num_cols])
        
        # Feature Engineering
        self.df['Interaction_1_3'] = self.df['Satisfaction'] * self.df['Salary']
        
        # Encoding
        self.df = pd.get_dummies(self.df, drop_first=True)
        
        # Split
        X = self.df.drop(self.target, axis=1)
        y = self.df[self.target]
        
        # Scaling
        X_scaled = pd.DataFrame(self.scaler.fit_transform(X), columns=X.columns)
        
        return train_test_split(X_scaled, y, test_size=0.2, random_state=42)

processor = DataPreprocessor(df, 'Left')
X_train, X_test, y_train, y_test = processor.process()
print('Data Processed. Train Shape:', X_train.shape)

In [ ]:
# Model Training & Comparison
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    results[name] = score
    print(f'{name} Accuracy: {score:.4f}')

best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
print(f'\nBest Model: {best_model_name}')

In [ ]:
# Detailed Evaluation
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print('Classification Report:\n')
print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix ({best_model_name})')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic')
axes[1].legend(loc='lower right')

plt.show()

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='importance', y='feature', data=importances.head(10), palette='viridis')
    plt.title('Top 10 Feature Importance')
    plt.show()

## 💡 Business Recommendations

Based on the analysis, we recommend:

1. **Target High-Risk Segments**: Focus retention efforts on customers with high `Satisfaction` and low tenure.
2. **Monitor Key Indicators**: The top features identified (e.g., `Satisfaction`) should be tracked on a dashboard.
3. **Intervention Strategy**: Implement proactive outreach for customers showing patterns similar to the 'Churn' class.